# Pipeline A: relevance x role (fixed formula)

Combines the Stage-1 relevance gatekeeper with the Stage-2 4-way role classifier using the original fixed multiplicative formula:

```
P(None | s)      = P(z=0 | s)
P(role | s)      = P(z=1 | s) * P(role | s, z=1)
```

No fitting involved — a deterministic combination of two independently-trained models' probabilities — so this only needs the held-out test set, not the validation set.

## 1. Setup

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

TEST_DF_PATH = Path("predictions/test_df.csv")
REGISTRY_PATH = Path("roberta_models/best_models_registry.json")
MODELS_DIR = Path("roberta_models")

TEXT_COL = "sent_text"
BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ALL_LABELS = ["None", "beoordeling", "beslissing", "materiele feiten", "proceshandelingen"]
LEGAL_LABELS = ["beoordeling", "beslissing", "materiele feiten", "proceshandelingen"]

print("Device:", DEVICE)

Device: cuda


## 2. Load test data

In [2]:
test_df = pd.read_csv(TEST_DF_PATH, keep_default_na=False)

print("Test rows:", len(test_df))
test_df["label"].value_counts()

Test rows: 1502


label
None                 541
beoordeling          389
materiele feiten     297
proceshandelingen    206
beslissing            69
Name: count, dtype: int64

## 3. Load Stage 1 and Stage 2 models

Resolved from `best_models_registry.json`, same pattern as `01_five_way_inference.ipynb`. Only these two models are needed for Pipeline A.

In [3]:
with open(REGISTRY_PATH, "r", encoding="utf-8") as f:
    registry = json.load(f)


def clean_col(label):
    return label.strip().lower().replace(" ", "_")


def resolve_model_path(model_info):
    return MODELS_DIR / Path(model_info["saved_path"]).name


def load_model(model_key):
    model_info = registry[model_key]
    model_path = resolve_model_path(model_info)

    if not model_path.exists():
        raise FileNotFoundError(f"Model path not found: {model_path}")

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    model.to(DEVICE)
    model.eval()

    labels = model_info.get("labels_order", model_info.get("labels"))
    max_len = model_info.get("max_len", 256)

    return tokenizer, model, labels, max_len


@torch.no_grad()
def predict_probs(texts, tokenizer, model, max_len, batch_size=BATCH_SIZE, desc=""):
    all_probs = []

    for start in tqdm(range(0, len(texts), batch_size), desc=desc):
        batch = texts[start:start + batch_size]

        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors="pt"
        )
        enc = {k: v.to(DEVICE) for k, v in enc.items()}

        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1)
        all_probs.append(probs.cpu().numpy())

    return np.vstack(all_probs)

## 4. Run inference

In [4]:
texts = test_df[TEXT_COL].fillna("").astype(str).tolist()

tokenizer1, model1, labels1, max_len1 = load_model("stage1_gatekeeper")
stage1_probs = predict_probs(texts, tokenizer1, model1, max_len1, desc="Stage 1 (relevance)")

test_df["p_none"] = stage1_probs[:, 0]
test_df["p_labelled"] = stage1_probs[:, 1]

del model1, tokenizer1
print("Stage 1 labels order:", labels1)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Stage 1 (relevance):   0%|          | 0/47 [00:00<?, ?it/s]

Stage 1 labels order: ['not_labelled', 'labelled']


In [5]:
tokenizer2, model2, labels2, max_len2 = load_model("stage2_text_only_4way")
stage2_probs = predict_probs(texts, tokenizer2, model2, max_len2, desc="Stage 2 (4-way role)")

for i, lab in enumerate(labels2):
    test_df[f"stage2_p_{clean_col(lab)}"] = stage2_probs[:, i]

del model2, tokenizer2
print("Stage 2 labels order:", labels2)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Stage 2 (4-way role):   0%|          | 0/47 [00:00<?, ?it/s]

Stage 2 labels order: ['beoordeling', 'beslissing', 'materiele feiten', 'proceshandelingen']


## 5. Save raw model outputs

In [6]:
OUTPUT_PATH = Path("predictions/pipeline_a_test_outputs.csv")
test_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("Saved:", OUTPUT_PATH)

Saved: predictions\pipeline_a_test_outputs.csv


## 6. Combine: Pipeline A formula

`P(None) = P(z=0)`, `P(role) = P(z=1) x P(role | z=1)`. Probabilistic chaining, not a hard gate — Stage 1's uncertainty is preserved and carried through the multiplication rather than being collapsed into a binary cutoff.

In [7]:
def make_pipeline_a(df):
    probs = pd.DataFrame(index=df.index)
    probs["None"] = df["p_none"]

    for label in LEGAL_LABELS:
        clean = clean_col(label)
        probs[label] = df["p_labelled"] * df[f"stage2_p_{clean}"]

    return probs


pipeline_a_probs = make_pipeline_a(test_df)
test_df["pipeline_a_pred_label"] = pipeline_a_probs.idxmax(axis=1)

pipeline_a_probs.head()

,None,beoordeling,beslissing,materiele feiten,proceshandelingen
0,0.666001,0.329681,0.000998,0.002611,0.000709
1,0.195918,0.797377,0.001017,0.003734,0.001954
2,0.028706,0.004702,0.001667,0.960022,0.004903
3,0.033778,0.005166,0.001305,0.954751,0.004999
4,0.946429,0.016866,0.001685,0.012790,0.022230


## 7. Evaluate

Sanity check: should reproduce the previously reported Pipeline A numbers (accuracy 0.6372, macro F1 0.6513) if the same checkpoints and test set are being used.

In [8]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support
)

y_true = test_df["label"]
y_pred = test_df["pipeline_a_pred_label"]

acc = accuracy_score(y_true, y_pred)
macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=ALL_LABELS, average="macro", zero_division=0
)

print(f"Pipeline A accuracy: {acc:.4f}")
print(f"Macro P: {macro_p:.4f}  Macro R: {macro_r:.4f}  Macro F1: {macro_f1:.4f}")
print()
print(classification_report(y_true, y_pred, labels=ALL_LABELS, digits=4, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=ALL_LABELS)
pd.DataFrame(
    cm,
    index=[f"true_{l}" for l in ALL_LABELS],
    columns=[f"pred_{l}" for l in ALL_LABELS]
)

Pipeline A accuracy: 0.6372
Macro P: 0.6615  Macro R: 0.6498  Macro F1: 0.6513

                   precision    recall  f1-score   support

             None     0.7403    0.6322    0.6820       541
      beoordeling     0.5706    0.7378    0.6435       389
       beslissing     0.8438    0.7826    0.8120        69
 materiele feiten     0.6331    0.5286    0.5761       297
proceshandelingen     0.5200    0.5680    0.5429       206

         accuracy                         0.6372      1502
        macro avg     0.6615    0.6498    0.6513      1502
     weighted avg     0.6497    0.6372    0.6380      1502



,pred_None,pred_beoordeling,pred_beslissing,pred_materiele feiten,pred_proceshandelingen
true_None,342,108,8,40,43
true_beoordeling,44,287,2,32,24
true_beslissing,6,7,54,2,0
true_materiele feiten,57,42,0,157,41
true_proceshandelingen,13,59,0,17,117


## 8. Stage 1 error analysis (relevance gatekeeper alone)

Stage 1 is a binary classifier: None vs. legally labelled. Its own argmax decision, independent of Stage 2 or the combination formula.

In [9]:
test_df["y_labelled"] = (test_df["label"] != "None").astype(int)
test_df["stage1_pred_idx"] = (test_df["p_labelled"] > test_df["p_none"]).astype(int)
test_df["stage1_correct"] = test_df["y_labelled"] == test_df["stage1_pred_idx"]

stage1_cm = confusion_matrix(test_df["y_labelled"], test_df["stage1_pred_idx"], labels=[0, 1])
print("Stage 1 confusion matrix (rows=true, cols=pred; 0=None, 1=labelled)")
display(pd.DataFrame(stage1_cm, index=["true_None", "true_labelled"], columns=["pred_None", "pred_labelled"]))

n_fp = ((test_df["y_labelled"] == 0) & (test_df["stage1_pred_idx"] == 1)).sum()
n_fn = ((test_df["y_labelled"] == 1) & (test_df["stage1_pred_idx"] == 0)).sum()
print(f"\nFalse positives (truly None, called relevant): {n_fp}")
print(f"False negatives (truly relevant, called None):  {n_fn}")
print(f"Stage 1 accuracy: {test_df['stage1_correct'].mean():.4f}")

Stage 1 confusion matrix (rows=true, cols=pred; 0=None, 1=labelled)


,pred_None,pred_labelled
true_None,329,212
true_labelled,110,851



False positives (truly None, called relevant): 212
False negatives (truly relevant, called None):  110
Stage 1 accuracy: 0.7856


In [10]:
test_df["stage1_confidence"] = test_df[["p_none", "p_labelled"]].max(axis=1)

conf_summary = test_df.groupby("stage1_correct")["stage1_confidence"].describe()[["count", "mean", "50%", "std"]]
conf_summary.index = ["incorrect", "correct"]
print("Stage 1 confidence: correct vs incorrect")
display(conf_summary.round(3))

hdr_err = (
    test_df.assign(is_error=~test_df["stage1_correct"])
    .groupby("hdr_group")["is_error"]
    .agg(n_sentences="count", n_errors="sum")
)
hdr_err["error_rate_pct"] = (100 * hdr_err["n_errors"] / hdr_err["n_sentences"]).round(2)
print("\nStage 1 error rate by header group")
display(hdr_err.sort_values("error_rate_pct", ascending=False))

case_err = (
    test_df.assign(is_error=~test_df["stage1_correct"])
    .groupby("case_name")["is_error"]
    .agg(n_sentences="count", n_errors="sum")
)
print(f"\nDocuments with zero Stage 1 errors: {(case_err['n_errors'] == 0).sum()} / {len(case_err)}")

Stage 1 confidence: correct vs incorrect


,count,mean,50%,std
incorrect,322.0,0.786,0.822,0.146
correct,1180.0,0.880,0.927,0.117



Stage 1 error rate by header group


,n_sentences,n_errors,error_rate_pct
hdr_group,,,
Context,49,15,30.61
Beoordeling,676,175,25.89
Feiten,288,67,23.26
Proceshandelingen partijen,252,43,17.06
Beslissing,237,22,9.28



Documents with zero Stage 1 errors: 2 / 20


In [11]:
N_EXAMPLES = 6

false_positives = test_df[(test_df["y_labelled"] == 0) & (test_df["stage1_pred_idx"] == 1)]
false_negatives = test_df[(test_df["y_labelled"] == 1) & (test_df["stage1_pred_idx"] == 0)]

print(f"False positives (n={len(false_positives)}) - sample:")
for _, row in false_positives.sample(min(N_EXAMPLES, len(false_positives)), random_state=42).iterrows():
    print(f"- [{row['case_name']} | hdr={row['hdr_group']} | p_labelled={row['p_labelled']:.2f}] {row['sent_text']}")

print(f"\nFalse negatives (n={len(false_negatives)}) - sample:")
for _, row in false_negatives.sample(min(N_EXAMPLES, len(false_negatives)), random_state=42).iterrows():
    print(f"- [{row['case_name']} | hdr={row['hdr_group']} | true={row['label']} | p_none={row['p_none']:.2f}] {row['sent_text']}")

False positives (n=212) - sample:
- [ECLI:NL:RBOBR:2020:3584.txt | hdr=Feiten | p_labelled=0.84] Opdrachtnemer oefent de Werkzaamheden gedurende gemiddeld veertig (40) uren per week uit (...); Artike
- [ECLI:NL:RBOBR:2015:1104.txt | hdr=Beslissing | p_labelled=0.91] a.v. feit 1 primair, feit 1 subsidiair: Niet-ontvankelijkverklaring van de benadeelde partij [slachtoffer 6] in de vordering.
- [ECLI:NL:RBOVE:2017:1505.txt | hdr=Beoordeling | p_labelled=0.96] Om de ernst van het werken volgens de normen en regels nogmaals te benadrukken is er een extra werkoverleg belegd op [datum] en op [datum] [verzoeker] heeft niet betwist dat hij genoemde e-mailberichten heeft ontvangen en dat hij aanwezig is geweest bij de extra werkoverleggen.
- [ECLI:NL:RBROT:2018:8513.txt | hdr=Proceshandelingen partijen | p_labelled=0.74] Bij TBS met dwangverpleging wordt de ter beschikking gestelde verplicht verpleegd in een gesloten inrichting.
- [ECLI:NL:RBNNE:2015:2927.txt | hdr=Beoordeling | p_labelled=0.91]

## 9. Stage 2 error analysis (4-way role classifier alone)

Stage 2's own argmax among the 4 legal roles, evaluated only on the 961 gold-relevant sentences — independent of Stage 1 and the combination formula.

In [12]:
stage2_prob_cols = {l: f"stage2_p_{clean_col(l)}" for l in LEGAL_LABELS}

stage2_probs_only = test_df[[stage2_prob_cols[l] for l in LEGAL_LABELS]].copy()
stage2_probs_only.columns = LEGAL_LABELS
test_df["stage2_only_pred_label"] = stage2_probs_only.idxmax(axis=1)

legal_df = test_df[test_df["label"].isin(LEGAL_LABELS)].copy()
legal_df["stage2_correct"] = legal_df["stage2_only_pred_label"] == legal_df["label"]

print(f"Stage 2 accuracy on gold-relevant subset: {legal_df['stage2_correct'].mean():.4f}")

stage2_errors = legal_df[~legal_df["stage2_correct"]]
support_by_role = legal_df["label"].value_counts()

confusion_pairs_s2 = (
    stage2_errors.groupby(["label", "stage2_only_pred_label"]).size().rename("count").reset_index()
)
confusion_pairs_s2["pct_of_true_label_support"] = confusion_pairs_s2.apply(
    lambda r: round(100 * r["count"] / support_by_role[r["label"]], 2), axis=1
)
confusion_pairs_s2 = confusion_pairs_s2.sort_values("count", ascending=False).reset_index(drop=True)

print("\nStage 2 confusion pairs (errors only)")
confusion_pairs_s2

Stage 2 accuracy on gold-relevant subset: 0.7315

Stage 2 confusion pairs (errors only)


,label,stage2_only_pred_label,count,pct_of_true_label_support
0,proceshandelingen,beoordeling,61,29.61
1,materiele feiten,proceshandelingen,56,18.86
2,materiele feiten,beoordeling,48,16.16
3,beoordeling,materiele feiten,36,9.25
4,beoordeling,proceshandelingen,27,6.94
5,proceshandelingen,materiele feiten,19,9.22
6,beslissing,beoordeling,7,10.14
7,beoordeling,beslissing,2,0.51
8,beslissing,materiele feiten,2,2.90


In [13]:
recall_matrix_s2 = pd.crosstab(
    legal_df["label"], legal_df["stage2_only_pred_label"], normalize="index"
) * 100
recall_matrix_s2 = recall_matrix_s2.reindex(index=LEGAL_LABELS, columns=LEGAL_LABELS, fill_value=0).round(2)
print("Stage 2 recall-normalized confusion matrix (%)")
display(recall_matrix_s2)

legal_df["stage2_confidence"] = stage2_probs_only.loc[legal_df.index].max(axis=1)
conf_summary_s2 = legal_df.groupby("stage2_correct")["stage2_confidence"].describe()[["count", "mean", "50%", "std"]]
conf_summary_s2.index = ["incorrect", "correct"]
print("\nStage 2 confidence: correct vs incorrect")
display(conf_summary_s2.round(3))

hdr_err_s2 = (
    legal_df.assign(is_error=~legal_df["stage2_correct"])
    .groupby("hdr_group")["is_error"]
    .agg(n_sentences="count", n_errors="sum")
)
hdr_err_s2["error_rate_pct"] = (100 * hdr_err_s2["n_errors"] / hdr_err_s2["n_sentences"]).round(2)
print("\nStage 2 error rate by header group")
display(hdr_err_s2.sort_values("error_rate_pct", ascending=False))

Stage 2 recall-normalized confusion matrix (%)


stage2_only_pred_label,beoordeling,beslissing,materiele feiten,proceshandelingen
label,,,,
beoordeling,83.29,0.51,9.25,6.94
beslissing,10.14,86.96,2.90,0.00
materiele feiten,16.16,0.00,64.98,18.86
proceshandelingen,29.61,0.00,9.22,61.17



Stage 2 confidence: correct vs incorrect


,count,mean,50%,std
incorrect,258.0,0.803,0.858,0.171
correct,703.0,0.922,0.975,0.114



Stage 2 error rate by header group


,n_sentences,n_errors,error_rate_pct
hdr_group,,,
Proceshandelingen partijen,177,60,33.90
Feiten,151,49,32.45
Beslissing,166,47,28.31
Beoordeling,449,100,22.27
Context,18,2,11.11


In [14]:
N_TOP_PAIRS = 4
N_EXAMPLES_PER_PAIR = 4

top_pairs_s2 = confusion_pairs_s2.head(N_TOP_PAIRS)[["label", "stage2_only_pred_label"]].itertuples(index=False)

for true_label, pred_label in top_pairs_s2:
    subset = legal_df[(legal_df["label"] == true_label) & (legal_df["stage2_only_pred_label"] == pred_label)]

    print("=" * 90)
    print(f"True: {true_label}  ->  Stage 2 predicted: {pred_label}   ({len(subset)} cases)")
    print("=" * 90)

    sample = subset.sample(min(N_EXAMPLES_PER_PAIR, len(subset)), random_state=42)
    for _, row in sample.iterrows():
        print(f"- [{row['case_name']} | hdr={row['hdr_group']} | conf={row['stage2_confidence']:.2f}] {row['sent_text']}")
    print()

True: proceshandelingen  ->  Stage 2 predicted: beoordeling   (61 cases)
- [ECLI:NL:GHAMS:2015:2960.txt | hdr=Beoordeling | conf=0.92] Daarin klaagt [appellante] dat de kantonrechter niet heeft aangenomen dat [geïntimeerde] een doos met voedingssupplementen heeft besteld en afgenomen van haar.
- [ECLI:NL:RBNNE:2016:4308.txt | hdr=Proceshandelingen partijen | conf=0.99] De clausule in het testament, in onderling verband en samenhang bezien met de overige bepalingen in het testament en de plaats en wijze waarop het testament tot stand is gebracht, kan niet anders dan tot de slotsom leiden dat sprake is van een uitsluitingsclausule als bedoel in artikel 1:94 lid s Burgerlijk Wetboek (BW).
- [ECLI:NL:RBDHA:2018:3316.txt | hdr=Proceshandelingen partijen | conf=0.99] De rechtbank wijst de vordering van de officier van justitie dan ook af.
- [ECLI:NL:GHARL:2015:6258.txt | hdr=Beoordeling | conf=0.99] SWV heeft haar grieven en de toelichting daarop in de door haar op [datum] betekende exploten

## 10. Where do Pipeline A's errors actually come from?

Pipeline A multiplies every one of Stage 2's 4 role probabilities by the *same* constant (`p_labelled`). Multiplying by a constant never changes which of the 4 is largest — so **whenever Pipeline A predicts a role at all, it is always exactly Stage 2's own argmax role.** Pipeline A can only ever disagree with Stage 2 by instead predicting None. That gives a clean, provable three-way split of every Pipeline A mistake on the 961 gold-relevant sentences:

- **Inherited from Stage 1** — Stage 1 itself said "None," so Pipeline A is mathematically forced to predict None regardless of what Stage 2 thinks.
- **Inherited from Stage 2** — Stage 1 said "relevant," but Stage 2's own argmax role was already wrong.
- **Combination-induced** — Stage 1 said "relevant" *and* Stage 2's argmax was the correct role, yet Pipeline A still predicts None, because `p_none` happened to exceed `p_labelled x P(role)` for the true role. This is the one failure mode that belongs to the combination formula itself, not to either component model.

In [15]:
# Confirm the claim empirically before relying on it: whenever Pipeline A predicts
# a role (not None), does it always match Stage 2's own argmax?

role_predicted_mask = test_df["pipeline_a_pred_label"] != "None"
mismatch = test_df[
    role_predicted_mask & (test_df["pipeline_a_pred_label"] != test_df["stage2_only_pred_label"])
]

print(f"Rows where Pipeline A predicts a role that differs from Stage 2's own argmax: {len(mismatch)}")
assert len(mismatch) == 0, "Unexpected: Pipeline A's role choice diverged from Stage 2's argmax"

Rows where Pipeline A predicts a role that differs from Stage 2's own argmax: 0


In [16]:
def decompose_pipeline_a_error(row):
    if row["label"] == row["pipeline_a_pred_label"]:
        return "pipeline_correct"
    if row["p_labelled"] <= row["p_none"]:
        return "inherited_from_stage1"
    if row["stage2_only_pred_label"] != row["label"]:
        return "inherited_from_stage2"
    return "combination_induced"


legal_df["pipeline_a_pred_label"] = test_df.loc[legal_df.index, "pipeline_a_pred_label"]
legal_df["pipeline_a_error_type"] = legal_df.apply(decompose_pipeline_a_error, axis=1)

decomposition = legal_df["pipeline_a_error_type"].value_counts()
decomposition_pct = (100 * decomposition / len(legal_df)).round(2)

print(f"Pipeline A error decomposition on the {len(legal_df)} gold-relevant test sentences")
pd.DataFrame({"count": decomposition, "pct_of_gold_relevant": decomposition_pct})

Pipeline A error decomposition on the 961 gold-relevant test sentences


,count,pct_of_gold_relevant
pipeline_a_error_type,,
pipeline_correct,615,64.00
inherited_from_stage2,230,23.93
inherited_from_stage1,110,11.45
combination_induced,6,0.62


In [17]:
# The interesting category: both component models individually "got it right"
# by their own argmax, yet the combination formula still flipped to None.

combo_induced = legal_df[legal_df["pipeline_a_error_type"] == "combination_induced"].copy()
combo_induced["margin"] = combo_induced["p_none"] - (
    combo_induced["p_labelled"] * combo_induced.apply(
        lambda r: r[stage2_prob_cols[r["label"]]], axis=1
    )
)

print(f"Combination-induced errors: {len(combo_induced)}")
if len(combo_induced) > 0:
    print("\nHow close was the flip? (p_none minus the true role's combined probability)")
    display(combo_induced["margin"].describe()[["mean", "50%", "min", "max"]].round(3))

    print("\nSample cases:")
    sample = combo_induced.sample(min(6, len(combo_induced)), random_state=42)
    for _, row in sample.iterrows():
        print(
            f"- [{row['case_name']} | hdr={row['hdr_group']} | true={row['label']} | "
            f"p_none={row['p_none']:.2f} vs p_labelled*P(role)={row['p_labelled'] * row[stage2_prob_cols[row['label']]]:.2f}] "
            f"{row['sent_text']}"
        )

Combination-induced errors: 6

How close was the flip? (p_none minus the true role's combined probability)


mean    0.102
50%     0.088
min     0.008
max     0.218
Name: margin, dtype: float64


Sample cases:
- [ECLI:NL:RBNNE:2016:4308.txt | hdr=Proceshandelingen partijen | true=proceshandelingen | p_none=0.47 vs p_labelled*P(role)=0.46] Met betrekking tot hen heeft de oom voorts bepaald dat bij vooroverlijden van een broer of (schoon)zus met achterlating van een echtgeno(o)t(e) deze echtgeno(o)t(e) voor de voor overleden broer of (schoon)zus in de plaats treedt.
- [ECLI:NL:RBOBR:2020:3584.txt | hdr=Feiten | true=beoordeling | p_none=0.48 vs p_labelled*P(role)=0.46] .
- [ECLI:NL:RBDHA:2021:6182.txt | hdr=Beoordeling | true=materiele feiten | p_none=0.36 vs p_labelled*P(role)=0.33] Het betreft een speciale regeling die voorgaat op de Wob (een lex specialis).
- [ECLI:NL:RBOBR:2020:3584.txt | hdr=Proceshandelingen partijen | true=proceshandelingen | p_none=0.48 vs p_labelled*P(role)=0.26] De Partner Compliance Statement dient slechts als instrument om bedreigingen van de onafhankelijkheid en kwaliteit van de accountant en de accountantsorganisatie te inventariseren.
- [ECLI:NL:G

## 11. Pipeline A vs. direct 5-way: relevance errors vs. role-confusion errors

Same split used in the 5-way model's error analysis (`01_five_way_inference.ipynb`, section 7.2), computed here for Pipeline A on the full 1502-sentence test set, so the two are directly comparable.

In [18]:
def classify_error(row, pred_col):
    if row["label"] == row[pred_col]:
        return "correct"
    if row["label"] == "None" or row[pred_col] == "None":
        return "relevance_error"
    return "role_confusion_error"


test_df["pipeline_a_error_type_5way"] = test_df.apply(classify_error, pred_col="pipeline_a_pred_label", axis=1)

pipeline_a_errors_only = test_df[test_df["pipeline_a_error_type_5way"] != "correct"]
pipeline_a_split = pipeline_a_errors_only["pipeline_a_error_type_5way"].value_counts()
pipeline_a_split_pct = (100 * pipeline_a_split / len(pipeline_a_errors_only)).round(2)

comparison = pd.DataFrame({
    "system": ["Direct 5-way", "Pipeline A"],
    "total_errors": [502, len(pipeline_a_errors_only)],
    "relevance_error_pct": [
        60.6,
        pipeline_a_split_pct.get("relevance_error", 0.0)
    ],
    "role_confusion_pct": [
        39.4,
        pipeline_a_split_pct.get("role_confusion_error", 0.0)
    ]
})

comparison

,system,total_errors,relevance_error_pct,role_confusion_pct
0,Direct 5-way,502,60.60,39.40
1,Pipeline A,545,58.53,41.47


## 11b. Document-level error clustering (Pipeline A's combined prediction)

Same methodology as `01_five_way_inference.ipynb` (section 7, document-level error clustering), computed here on Pipeline A's final combined prediction (`pipeline_a_pred_label`) rather than a single component model's own errors, so the two are directly comparable.

In [19]:
pipeline_a_case_error_counts = (
    test_df
    .assign(is_error=test_df["label"] != test_df["pipeline_a_pred_label"])
    .groupby("case_name")["is_error"]
    .agg(n_sentences="count", n_errors="sum")
)
pipeline_a_case_error_counts["error_rate_pct"] = (
    100 * pipeline_a_case_error_counts["n_errors"] / pipeline_a_case_error_counts["n_sentences"]
).round(2)

print(f"Test-set documents: {len(pipeline_a_case_error_counts)}")
print(f"Documents with zero errors: {(pipeline_a_case_error_counts['n_errors'] == 0).sum()}")

sorted_by_errors = pipeline_a_case_error_counts.sort_values("n_errors", ascending=False)
total_errors = sorted_by_errors["n_errors"].sum()
cum_pct = 100 * sorted_by_errors["n_errors"].cumsum() / total_errors

print("\nCumulative share of all errors covered by the top-N error-heaviest documents:")
for n in [5, 10, 20]:
    print(f"  top {n} documents: {cum_pct.iloc[n - 1]:.1f}%")

print("\nTop 10 documents by error count:")
sorted_by_errors.head(10)

Test-set documents: 20
Documents with zero errors: 2

Cumulative share of all errors covered by the top-N error-heaviest documents:
  top 5 documents: 51.2%
  top 10 documents: 79.8%
  top 20 documents: 100.0%

Top 10 documents by error count:


,n_sentences,n_errors,error_rate_pct
case_name,,,
ECLI:NL:RBOBR:2020:3584.txt,221,74,33.48
ECLI:NL:RBOVE:2017:1505.txt,129,61,47.29
ECLI:NL:HR:2018:874.txt,108,51,47.22
ECLI:NL:RBDHA:2018:3316.txt,121,48,39.67
ECLI:NL:RBNNE:2015:2927.txt,99,45,45.45
ECLI:NL:RBOBR:2016:6963.txt,123,37,30.08
ECLI:NL:GHARL:2017:8427.txt,148,32,21.62
ECLI:NL:RBNNE:2016:4308.txt,77,32,41.56
ECLI:NL:RBDHA:2021:6182.txt,76,28,36.84


## 12. Header-prior Bayesian fusion

Header fusion is applied independently to Stage 1 and Stage 2 — each gets its own header prior and its own validation-tuned λ (see `06_stage1_relevance.ipynb` and `07_stage2_fourway.ipynb` for the component-level results this reuses) — and the two fused outputs are then recombined through the same Pipeline A formula used in section 6.

In [20]:
TRAIN_DF_PATH = Path("predictions/train_df.csv")
EVAL_DF_PATH = Path("predictions/eval_df.csv")
ALPHA = 1.0
BINARY_CLASSES = [0, 1]

train_df = pd.read_csv(TRAIN_DF_PATH, keep_default_na=False)
eval_df = pd.read_csv(EVAL_DF_PATH, keep_default_na=False)
train_df["y_labelled"] = (train_df["label"] != "None").astype(int)
eval_df["y_labelled"] = (eval_df["label"] != "None").astype(int)
train_legal = train_df[train_df["label"].isin(LEGAL_LABELS)].copy()
eval_legal = eval_df[eval_df["label"].isin(LEGAL_LABELS)].copy()


def make_header_prior(df_train, label_col, labels, alpha=1.0):
    counts = (
        df_train.groupby(["hdr_group", label_col]).size()
        .unstack(fill_value=0)
        .reindex(columns=labels, fill_value=0)
    )
    probs = counts + alpha
    return probs.div(probs.sum(axis=1), axis=0)


def make_global_prior(df_train, label_col, labels, alpha=1.0):
    counts = df_train[label_col].value_counts().reindex(labels, fill_value=0) + alpha
    return (counts / counts.sum()).values


def get_meta_prior_df(df_apply, header_prior_train, global_prior, labels):
    def get_prior(hdr_group):
        if hdr_group in header_prior_train.index:
            return header_prior_train.loc[hdr_group].values
        return global_prior

    mat = np.vstack(df_apply["hdr_group"].apply(get_prior))
    return pd.DataFrame(mat, columns=labels, index=df_apply.index)


def combine_log_scores(text_probs, metadata_probs, lam, eps=1e-12):
    text = np.clip(text_probs.values, eps, 1.0)
    metadata = np.clip(metadata_probs.values, eps, 1.0)

    scores = np.log(text) + lam * np.log(metadata)
    scores -= scores.max(axis=1, keepdims=True)
    scores = np.exp(scores)
    scores /= scores.sum(axis=1, keepdims=True)

    return pd.DataFrame(scores, columns=text_probs.columns, index=text_probs.index)


LAMBDA_GRID = [0, 0.1, 0.25, 0.5, 1, 2, 3, 5]

print("Train rows:", len(train_df), " | Validation rows:", len(eval_df))

Train rows: 5608  | Validation rows: 1281


### 12.1 Tune λ for Stage 1

In [21]:
tok1, mdl1, lbl1, ml1 = load_model("stage1_gatekeeper")
eval_stage1_probs = predict_probs(
    eval_df[TEXT_COL].fillna("").astype(str).tolist(), tok1, mdl1, ml1, BATCH_SIZE, desc="[val] stage1"
)
del mdl1, tok1
if DEVICE == "cuda":
    torch.cuda.empty_cache()

eval_stage1_text_probs = pd.DataFrame(eval_stage1_probs, columns=BINARY_CLASSES, index=eval_df.index)

header_prior_s1 = make_header_prior(train_df, "y_labelled", BINARY_CLASSES, ALPHA)
global_prior_s1 = make_global_prior(train_df, "y_labelled", BINARY_CLASSES, ALPHA)
eval_meta_prior_s1 = get_meta_prior_df(eval_df, header_prior_s1, global_prior_s1, BINARY_CLASSES)

sweep_s1 = []
for lam in LAMBDA_GRID:
    fused = combine_log_scores(eval_stage1_text_probs, eval_meta_prior_s1, lam)
    pred = fused.idxmax(axis=1)
    _, _, f1, _ = precision_recall_fscore_support(
        eval_df["y_labelled"], pred, labels=BINARY_CLASSES, average="macro", zero_division=0
    )
    sweep_s1.append({"lambda": lam, "val_macro_f1": round(f1, 4)})

sweep_s1_df = pd.DataFrame(sweep_s1)
best_lambda_s1 = sweep_s1_df.loc[sweep_s1_df["val_macro_f1"].idxmax(), "lambda"]
print(sweep_s1_df)
print(f"Best Stage 1 lambda: {best_lambda_s1}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[val] stage1:   0%|          | 0/41 [00:00<?, ?it/s]

   lambda  val_macro_f1
0    0.00        0.7648
1    0.10        0.7614
2    0.25        0.7557
3    0.50        0.7439
4    1.00        0.7100
5    2.00        0.6578
6    3.00        0.6229
7    5.00        0.5191
Best Stage 1 lambda: 0.0


### 12.2 Tune λ for Stage 2

In [22]:
tok2, mdl2, lbl2, ml2 = load_model("stage2_text_only_4way")
eval_stage2_probs = predict_probs(
    eval_legal[TEXT_COL].fillna("").astype(str).tolist(), tok2, mdl2, ml2, BATCH_SIZE, desc="[val] stage2"
)
del mdl2, tok2
if DEVICE == "cuda":
    torch.cuda.empty_cache()

eval_stage2_text_probs = pd.DataFrame(eval_stage2_probs, columns=lbl2, index=eval_legal.index)[LEGAL_LABELS]

header_prior_s2 = make_header_prior(train_legal, "label", LEGAL_LABELS, ALPHA)
global_prior_s2 = make_global_prior(train_legal, "label", LEGAL_LABELS, ALPHA)
eval_meta_prior_s2 = get_meta_prior_df(eval_legal, header_prior_s2, global_prior_s2, LEGAL_LABELS)

sweep_s2 = []
for lam in LAMBDA_GRID:
    fused = combine_log_scores(eval_stage2_text_probs, eval_meta_prior_s2, lam)
    pred = fused.idxmax(axis=1)
    _, _, f1, _ = precision_recall_fscore_support(
        eval_legal["label"], pred, labels=LEGAL_LABELS, average="macro", zero_division=0
    )
    sweep_s2.append({"lambda": lam, "val_macro_f1": round(f1, 4)})

sweep_s2_df = pd.DataFrame(sweep_s2)
best_lambda_s2 = sweep_s2_df.loc[sweep_s2_df["val_macro_f1"].idxmax(), "lambda"]
print(sweep_s2_df)
print(f"Best Stage 2 lambda: {best_lambda_s2}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[val] stage2:   0%|          | 0/24 [00:00<?, ?it/s]

   lambda  val_macro_f1
0    0.00        0.7827
1    0.10        0.7850
2    0.25        0.7914
3    0.50        0.7960
4    1.00        0.7717
5    2.00        0.7642
6    3.00        0.7690
7    5.00        0.7138
Best Stage 2 lambda: 0.5


### 12.3 Apply both fused stages to the test set and recombine via Pipeline A

In [23]:
test_stage1_text_probs = test_df[["p_none", "p_labelled"]].copy()
test_stage1_text_probs.columns = BINARY_CLASSES
test_meta_prior_s1 = get_meta_prior_df(test_df, header_prior_s1, global_prior_s1, BINARY_CLASSES)

test_stage2_text_probs_full = test_df[[f"stage2_p_{clean_col(l)}" for l in LEGAL_LABELS]].copy()
test_stage2_text_probs_full.columns = LEGAL_LABELS
test_meta_prior_s2_full = get_meta_prior_df(test_df, header_prior_s2, global_prior_s2, LEGAL_LABELS)

pipeline_a_header_results = []

for lam_s1, lam_s2, name in [
    (0, 0, "no fusion"),
    (1, 1, "both lambda=1"),
    (best_lambda_s1, best_lambda_s2, f"tuned (s1={best_lambda_s1}, s2={best_lambda_s2})")
]:
    fused_s1 = combine_log_scores(test_stage1_text_probs, test_meta_prior_s1, lam_s1)
    fused_s2 = combine_log_scores(test_stage2_text_probs_full, test_meta_prior_s2_full, lam_s2)

    final_probs = pd.DataFrame(index=test_df.index)
    final_probs["None"] = fused_s1[0]
    for label in LEGAL_LABELS:
        final_probs[label] = fused_s1[1] * fused_s2[label]

    pred = final_probs.idxmax(axis=1)

    acc = accuracy_score(test_df["label"], pred)
    _, _, f1, _ = precision_recall_fscore_support(
        test_df["label"], pred, labels=ALL_LABELS, average="macro", zero_division=0
    )
    pipeline_a_header_results.append({
        "setting": name, "lambda_stage1": lam_s1, "lambda_stage2": lam_s2,
        "test_accuracy": round(acc, 4), "test_macro_f1": round(f1, 4)
    })

print(pd.DataFrame(pipeline_a_header_results))

                  setting  lambda_stage1  lambda_stage2  test_accuracy  \
0               no fusion            0.0            0.0         0.6372   
1           both lambda=1            1.0            1.0         0.6451   
2  tuned (s1=0.0, s2=0.5)            0.0            0.5         0.6458   

   test_macro_f1  
0         0.6513  
1         0.6613  
2         0.6609  


In [24]:
print(f"Full classification report at tuned lambda (s1={best_lambda_s1}, s2={best_lambda_s2}) (test set)")
print(classification_report(test_df["label"], pred, labels=ALL_LABELS, digits=4, zero_division=0))

Full classification report at tuned lambda (s1=0.0, s2=0.5) (test set)
                   precision    recall  f1-score   support

             None     0.7370    0.6266    0.6773       541
      beoordeling     0.5787    0.7558    0.6555       389
       beslissing     0.8438    0.7826    0.8120        69
 materiele feiten     0.6494    0.5488    0.5949       297
proceshandelingen     0.5479    0.5825    0.5647       206

         accuracy                         0.6458      1502
        macro avg     0.6714    0.6593    0.6609      1502
     weighted avg     0.6577    0.6458    0.6461      1502



## 13. Confidence thresholding (None-first decision rule)

Same idea as `01_five_way_inference.ipynb` section 9: if `P(None)` from Stage 1 is `≥ θ_none`, predict None; otherwise predict Stage 2's own argmax among the 4 roles (still mutually exclusive — one clear winner).

`θ_none` is tuned on the training set, but the objective is Stage 1's **own None-class binary F1** in isolation — not the downstream Pipeline A macro F1 — matching the same per-class-only tuning philosophy used for the 5-way model and (already) for each OvR detector. The precision-maximizing threshold is shown alongside for comparison, and validation-set performance at the chosen threshold is reported as a transparency check.

In [25]:
tok1, mdl1, lbl1, ml1 = load_model("stage1_gatekeeper")
train_stage1_probs = predict_probs(
    train_df[TEXT_COL].fillna("").astype(str).tolist(), tok1, mdl1, ml1, BATCH_SIZE, desc="[train] stage1"
)
del mdl1, tok1
if DEVICE == "cuda":
    torch.cuda.empty_cache()

train_df["p_none"] = train_stage1_probs[:, 0]
train_df["p_labelled"] = train_stage1_probs[:, 1]

# Note: Stage 2 is not needed on the training set here — the threshold is tuned purely
# on Stage 1's own None-class separation, independent of Stage 2's downstream behavior.
print("Train rows scored:", len(train_df))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[train] stage1:   0%|          | 0/176 [00:00<?, ?it/s]

Train rows scored: 5608


In [26]:
def none_first_decision_pipeline_a(p_none, role_argmax, theta_none):
    return pd.Series(np.where(p_none >= theta_none, "None", role_argmax), index=p_none.index)


train_is_none = (train_df["label"] == "None").astype(int)

THETA_GRID = np.round(np.arange(0.05, 1.0, 0.05), 2)
sweep_results = []

for theta in THETA_GRID:
    pred_is_none = (train_df["p_none"] >= theta).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        train_is_none, pred_is_none, labels=[0, 1], average="binary", pos_label=1, zero_division=0
    )
    sweep_results.append({
        "theta_none": theta,
        "none_precision": round(precision, 4),
        "none_recall": round(recall, 4),
        "none_f1": round(f1, 4)
    })

sweep_df = pd.DataFrame(sweep_results)
best_theta_f1 = sweep_df.loc[sweep_df["none_f1"].idxmax(), "theta_none"]
best_theta_precision = sweep_df.loc[sweep_df["none_precision"].idxmax(), "theta_none"]
best_theta_none = best_theta_f1

print(sweep_df.to_string(index=False))
print(f"\nBest theta_none maximizing None-class F1: {best_theta_f1}")
print(f"Best theta_none maximizing None-class precision: {best_theta_precision}")
print(f"Using F1-maximizing threshold going forward: {best_theta_none}")

 theta_none  none_precision  none_recall  none_f1
       0.05          0.5107       0.9715   0.6694
       0.10          0.6363       0.9349   0.7573
       0.15          0.7124       0.9029   0.7964
       0.20          0.7556       0.8778   0.8121
       0.25          0.8008       0.8518   0.8255
       0.30          0.8340       0.8252   0.8296
       0.35          0.8515       0.7982   0.8240
       0.40          0.8701       0.7717   0.8179
       0.45          0.8828       0.7471   0.8093
       0.50          0.8933       0.7211   0.7980
       0.55          0.9026       0.6915   0.7831
       0.60          0.9098       0.6620   0.7664
       0.65          0.9196       0.6355   0.7516
       0.70          0.9285       0.6044   0.7322
       0.75          0.9344       0.5709   0.7087
       0.80          0.9425       0.5418   0.6881
       0.85          0.9536       0.4832   0.6414
       0.90          0.9647       0.4106   0.5760
       0.95          0.9682       0.2899   0.4462


### 13.1 Apply the tuned threshold to validation (transparency check) and test

In [27]:
# Stage 2 on the FULL validation set (section 12 only scored the legal-relevant subset)
tok2, mdl2, lbl2, ml2 = load_model("stage2_text_only_4way")
eval_stage2_probs_full = predict_probs(
    eval_df[TEXT_COL].fillna("").astype(str).tolist(), tok2, mdl2, ml2, BATCH_SIZE, desc="[val-full] stage2"
)
del mdl2, tok2
if DEVICE == "cuda":
    torch.cuda.empty_cache()

eval_stage2_argmax_full = pd.DataFrame(
    eval_stage2_probs_full, columns=lbl2, index=eval_df.index
)[LEGAL_LABELS].idxmax(axis=1)

eval_p_none_full = eval_stage1_text_probs[0]

val_pred_threshold = none_first_decision_pipeline_a(eval_p_none_full, eval_stage2_argmax_full, best_theta_none)
_, _, val_f1, _ = precision_recall_fscore_support(
    eval_df["label"], val_pred_threshold, labels=ALL_LABELS, average="macro", zero_division=0
)
print(f"[Transparency check] Validation macro F1 at theta_none={best_theta_none}: {val_f1:.4f}")

if "stage2_only_pred_label" not in test_df.columns:
    test_stage2_probs_only = test_df[[f"stage2_p_{clean_col(l)}" for l in LEGAL_LABELS]].copy()
    test_stage2_probs_only.columns = LEGAL_LABELS
    test_df["stage2_only_pred_label"] = test_stage2_probs_only.idxmax(axis=1)

test_pred_threshold = none_first_decision_pipeline_a(test_df["p_none"], test_df["stage2_only_pred_label"], best_theta_none)

acc = accuracy_score(test_df["label"], test_pred_threshold)
_, _, f1, _ = precision_recall_fscore_support(
    test_df["label"], test_pred_threshold, labels=ALL_LABELS, average="macro", zero_division=0
)

print(f"\nConfidence-thresholded Pipeline A (theta_none={best_theta_none}) on test set")
print(f"Accuracy: {acc:.4f}  Macro F1: {f1:.4f}")
print()
print(classification_report(test_df["label"], test_pred_threshold, labels=ALL_LABELS, digits=4, zero_division=0))

final_comparison = pd.DataFrame([
    {"system": "Plain formula (no fusion)", "test_accuracy": 0.6372, "test_macro_f1": 0.6513},
    {"system": "Header fusion (tuned)", "test_accuracy": 0.6458, "test_macro_f1": 0.6609},
    {"system": f"Confidence threshold (theta_none={best_theta_none})", "test_accuracy": round(acc, 4), "test_macro_f1": round(f1, 4)},
])
print(final_comparison)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[val-full] stage2:   0%|          | 0/41 [00:00<?, ?it/s]

[Transparency check] Validation macro F1 at theta_none=0.3: 0.6770

Confidence-thresholded Pipeline A (theta_none=0.3) on test set
Accuracy: 0.6365  Macro F1: 0.6390

                   precision    recall  f1-score   support

             None     0.6826    0.7116    0.6968       541
      beoordeling     0.6004    0.6915    0.6428       389
       beslissing     0.8333    0.7246    0.7752        69
 materiele feiten     0.6406    0.4680    0.5409       297
proceshandelingen     0.5305    0.5485    0.5394       206

         accuracy                         0.6365      1502
        macro avg     0.6575    0.6289    0.6390      1502
     weighted avg     0.6391    0.6365    0.6340      1502

                                  system  test_accuracy  test_macro_f1
0              Plain formula (no fusion)         0.6372         0.6513
1                  Header fusion (tuned)         0.6458         0.6609
2  Confidence threshold (theta_none=0.3)         0.6365         0.6390
